In [ ]:
"""
excel_to_ppt_charts.py
======================
Automates copying named charts from Excel files into PowerPoint templates.

Mapping file (mapping.xlsx) columns:
  - excel_file_name : Excel workbook filename (searched in current directory)
  - ppt_file_name   : PowerPoint filename     (searched in current directory)
  - excel           : Chart/shape name as it appears in Excel Selection Pane
  - ppt             : Shape name as it appears in PPT Selection Pane (target placeholder)

Requirements:
  pip install pandas openpyxl python-pptx pywin32 Pillow

Usage:
  Place this script in the same folder as your Excel/PPT files and mapping.xlsx,
  then run:  python excel_to_ppt_charts.py
"""

import os
import sys
import logging
import tempfile
import traceback
from pathlib import Path
from datetime import datetime

import pandas as pd


# ---------------------------------------------------------------------------
# Logging setup  (console + logs.txt in the script's working directory)
# ---------------------------------------------------------------------------
SCRIPT_DIR = Path.cwd()
LOG_FILE   = SCRIPT_DIR / "logs.txt"

def setup_logging() -> logging.Logger:
    logger = logging.getLogger("excel_to_ppt")
    logger.setLevel(logging.DEBUG)

    fmt = logging.Formatter("%(asctime)s  [%(levelname)-8s]  %(message)s",
                            datefmt="%Y-%m-%d %H:%M:%S")

    # File handler – always INFO+
    fh = logging.FileHandler(LOG_FILE, mode="w", encoding="utf-8")
    fh.setLevel(logging.DEBUG)
    fh.setFormatter(fmt)

    # Console handler – INFO+
    ch = logging.StreamHandler(sys.stdout)
    ch.setLevel(logging.INFO)
    ch.setFormatter(fmt)

    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger

log = setup_logging()


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def find_file(filename: str, search_dir: Path) -> Path | None:
    """Return the first file matching *filename* in *search_dir* (non-recursive first)."""
    direct = search_dir / filename
    if direct.exists():
        return direct.resolve()
    # Fallback: recursive search
    matches = list(search_dir.rglob(filename))
    if matches:
        log.warning(f"  '{filename}' not in root – found at: {matches[0]}")
        return matches[0].resolve()
    return None


def read_mapping(mapping_path: Path) -> pd.DataFrame:
    """Read and validate mapping.xlsx. Returns normalised DataFrame."""
    df = pd.read_excel(mapping_path, dtype=str)
    # Normalise column headers (strip, lowercase, underscores)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df = df.dropna(how="all")

    required = {"excel_file_name", "ppt_file_name", "excel", "ppt"}
    missing = required - set(df.columns)
    if missing:
        log.error(f"mapping.xlsx is missing columns: {missing}")
        log.error(f"Found columns: {list(df.columns)}")
        sys.exit(1)

    # Strip whitespace from all cells
    for col in required:
        df[col] = df[col].str.strip()

    df = df.dropna(subset=list(required))
    log.info(f"Mapping loaded – {len(df)} row(s) after cleaning")
    return df


# ---------------------------------------------------------------------------
# Excel chart export  (win32com / COM automation)
# ---------------------------------------------------------------------------
def get_excel_app():
    """Return a hidden, alert-suppressed Excel COM instance."""
    try:
        import win32com.client
        import pythoncom
        pythoncom.CoInitialize()
        app = win32com.client.Dispatch("Excel.Application")
        app.Visible          = False
        app.DisplayAlerts    = False
        app.ScreenUpdating   = False
        return app
    except ImportError:
        log.error("pywin32 is not installed.  Run:  pip install pywin32")
        sys.exit(1)
    except Exception as exc:
        log.error(f"Cannot start Excel COM: {exc}")
        sys.exit(1)


def export_chart_from_excel(excel_app, wb_path: Path, chart_name: str,
                             out_png: Path) -> bool:
    """
    Open *wb_path*, find a shape/chart named *chart_name* (Selection Pane name),
    export it as a PNG to *out_png*.  Returns True on success.
    """
    wb = None
    try:
        wb = excel_app.Workbooks.Open(str(wb_path), ReadOnly=True, UpdateLinks=False)

        for sheet in wb.Sheets:
            try:
                shapes = sheet.Shapes
            except Exception:
                continue

            for shape in shapes:
                if shape.Name.strip() == chart_name:
                    log.debug(f"    Found '{chart_name}' on sheet '{sheet.Name}'")

                    # ---- Try Chart.Export (works for ChartObject) ----
                    try:
                        shape.Chart.Export(str(out_png), "PNG")
                        log.info(f"    Exported chart '{chart_name}' → {out_png.name}")
                        return True
                    except Exception:
                        pass  # not a chart object – fall through

                    # ---- Fallback: copy as picture → paste to temp workbook → save ----
                    try:
                        shape.CopyPicture(Appearance=1, Format=2)   # xlScreen, xlBitmap
                        tmp_wb = excel_app.Workbooks.Add()
                        tmp_sheet = tmp_wb.Sheets(1)
                        tmp_sheet.Paste()

                        # The pasted picture lands as a shape
                        pasted = tmp_sheet.Shapes(tmp_sheet.Shapes.Count)
                        pasted.Export(str(out_png))       # may not work on all builds
                        tmp_wb.Close(SaveChanges=False)
                        log.info(f"    Exported shape '{chart_name}' via clipboard → {out_png.name}")
                        return True
                    except Exception as e2:
                        log.error(f"    Clipboard export also failed for '{chart_name}': {e2}")
                        try:
                            tmp_wb.Close(SaveChanges=False)
                        except Exception:
                            pass
                        return False

        log.warning(f"    Chart/shape '{chart_name}' NOT FOUND in '{wb_path.name}'")
        return False

    except Exception as exc:
        log.error(f"    Error opening workbook '{wb_path.name}': {exc}")
        log.debug(traceback.format_exc())
        return False

    finally:
        if wb is not None:
            try:
                wb.Close(SaveChanges=False)
            except Exception:
                pass


# ---------------------------------------------------------------------------
# PowerPoint manipulation  (python-pptx)
# ---------------------------------------------------------------------------
def load_pptx(ppt_path: Path):
    try:
        from pptx import Presentation
        return Presentation(str(ppt_path))
    except ImportError:
        log.error("python-pptx is not installed.  Run:  pip install python-pptx")
        sys.exit(1)
    except Exception as exc:
        log.error(f"Cannot open PPT '{ppt_path.name}': {exc}")
        return None


def build_shape_map(prs) -> dict:
    """
    Return  { shape_name: (slide_index_0based, slide, shape) }
    across all slides.  If a name appears on multiple slides, the last wins
    (log a warning).
    """
    shape_map = {}
    for s_idx, slide in enumerate(prs.slides):
        for shape in slide.shapes:
            name = shape.name.strip()
            if name in shape_map:
                log.warning(f"  Duplicate shape name '{name}' found – "
                            f"slide {shape_map[name][0]+1} and slide {s_idx+1}; "
                            f"using slide {s_idx+1}")
            shape_map[name] = (s_idx, slide, shape)
    return shape_map


def replace_shape_with_image(slide, shape, img_path: Path) -> bool:
    """
    Insert *img_path* at the exact position/size of *shape*, preserve z-order,
    then remove the original shape.  The inserted picture gets the same name.
    """
    from pptx.util import Emu   # noqa: F401 (imported to confirm package exists)

    left   = shape.left
    top    = shape.top
    width  = shape.width
    height = shape.height
    name   = shape.name

    sp_tree  = slide.shapes._spTree
    old_elem = shape._element

    # Remember z-order position (index inside spTree)
    children = list(sp_tree)
    try:
        z_idx = children.index(old_elem)
    except ValueError:
        z_idx = len(children)  # append at end

    # Remove the placeholder shape
    sp_tree.remove(old_elem)

    # Add the picture
    pic_shape = slide.shapes.add_picture(str(img_path), left, top, width, height)
    pic_shape.name = name   # restore original name

    # Restore z-order
    pic_elem = pic_shape._element
    sp_tree.remove(pic_elem)
    sp_tree.insert(z_idx, pic_elem)

    log.info(
        f"    ✓  Replaced '{name}' on slide {slide.slide_id} "
        f"[L={left//914400:.3f}\" T={top//914400:.3f}\" "
        f"W={width//914400:.3f}\" H={height//914400:.3f}\"]"
    )
    return True


# ---------------------------------------------------------------------------
# Main orchestrator
# ---------------------------------------------------------------------------
def main():
    log.info("=" * 70)
    log.info("Excel → PowerPoint Chart Automation")
    log.info(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"Workdir : {SCRIPT_DIR}")
    log.info("=" * 70)

    # ── 1. Read mapping ──────────────────────────────────────────────────────
    mapping_path = find_file("mapping.xlsx", SCRIPT_DIR)
    if not mapping_path:
        log.error("mapping.xlsx not found in the current directory – aborting.")
        sys.exit(1)
    log.info(f"Mapping : {mapping_path}")
    df = read_mapping(mapping_path)

    # ── 2. Start Excel COM ───────────────────────────────────────────────────
    excel_app = get_excel_app()
    log.info("Excel COM started (hidden)")

    # ── 3. Process each PPT file ─────────────────────────────────────────────
    total_ok   = 0
    total_fail = 0

    with tempfile.TemporaryDirectory(prefix="chart_export_") as tmp_dir:
        tmp_path = Path(tmp_dir)

        for ppt_filename, ppt_group in df.groupby("ppt_file_name"):
            log.info("")
            log.info(f"▶  PPT file : {ppt_filename}")

            # Locate PPT
            ppt_path = find_file(ppt_filename, SCRIPT_DIR)
            if not ppt_path:
                log.error(f"   PPT file NOT FOUND: {ppt_filename} – skipping {len(ppt_group)} row(s)")
                total_fail += len(ppt_group)
                continue

            # Load presentation
            prs = load_pptx(ppt_path)
            if prs is None:
                total_fail += len(ppt_group)
                continue

            shape_map = build_shape_map(prs)
            log.info(f"   Loaded PPT – {len(prs.slides)} slide(s), "
                     f"{len(shape_map)} named shape(s)")

            ppt_ok   = 0
            ppt_fail = 0

            for _, row in ppt_group.iterrows():
                excel_file   = row["excel_file_name"]
                excel_chart  = row["excel"]   # name in Excel Selection Pane
                ppt_shape    = row["ppt"]     # name in PPT  Selection Pane

                log.info(f"   ── Mapping: [{excel_file}] '{excel_chart}'"
                         f"  →  PPT shape '{ppt_shape}'")

                # Locate Excel file
                excel_path = find_file(excel_file, SCRIPT_DIR)
                if not excel_path:
                    log.error(f"      Excel file NOT FOUND: {excel_file}")
                    ppt_fail += 1
                    continue

                # Sanitise chart name for filename
                safe_name = "".join(c if c.isalnum() or c in "-_" else "_"
                                    for c in excel_chart)
                img_path  = tmp_path / f"{safe_name}.png"

                # Export chart from Excel
                exported = export_chart_from_excel(excel_app, excel_path,
                                                   excel_chart, img_path)
                if not exported:
                    log.error(f"      Chart export FAILED for '{excel_chart}'")
                    ppt_fail += 1
                    continue

                # Verify the image was created
                if not img_path.exists() or img_path.stat().st_size == 0:
                    log.error(f"      Exported image is missing/empty: {img_path}")
                    ppt_fail += 1
                    continue

                # Find target shape in PPT
                if ppt_shape not in shape_map:
                    log.error(f"      Shape '{ppt_shape}' NOT FOUND in {ppt_filename}")
                    log.debug(f"      Available shapes: {list(shape_map.keys())}")
                    ppt_fail += 1
                    continue

                s_idx, slide, shape = shape_map[ppt_shape]
                log.info(f"      Target on slide {s_idx + 1}: '{ppt_shape}'")

                # Replace
                try:
                    ok = replace_shape_with_image(slide, shape, img_path)
                    if ok:
                        # Update shape_map entry so the new picture is tracked
                        shape_map.pop(ppt_shape, None)
                        ppt_ok += 1
                    else:
                        ppt_fail += 1
                except Exception as exc:
                    log.error(f"      Replace FAILED: {exc}")
                    log.debug(traceback.format_exc())
                    ppt_fail += 1

            # Save the modified presentation
            try:
                prs.save(str(ppt_path))
                log.info(f"   Saved  : {ppt_path}")
            except Exception as exc:
                log.error(f"   Save FAILED for '{ppt_filename}': {exc}")
                ppt_fail += ppt_ok  # all replacements are lost
                ppt_ok = 0

            log.info(f"   Result : {ppt_ok} succeeded, {ppt_fail} failed")
            total_ok   += ppt_ok
            total_fail += ppt_fail

    # ── 4. Cleanup ────────────────────────────────────────────────────────────
    try:
        excel_app.Quit()
        log.info("")
        log.info("Excel COM closed cleanly")
    except Exception:
        pass

    # ── 5. Summary ────────────────────────────────────────────────────────────
    log.info("")
    log.info("=" * 70)
    log.info(f"SUMMARY  |  Succeeded: {total_ok}   Failed: {total_fail}   "
             f"Total rows: {total_ok + total_fail}")
    log.info(f"Finished : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    log.info(f"Log file : {LOG_FILE}")
    log.info("=" * 70)

    sys.exit(0 if total_fail == 0 else 1)


if __name__ == "__main__":
    main()